Missão 1: Análise Exploratória e Hipóteses

A Análise Exploratória de Dados (EDA) é o alicerce fundamental de qualquer pipeline preditivo robusto. Seguindo o rigor metodológico de O Preço da Precisão, restringimos nossa investigação inicial estritamente ao conjunto de treino (01/01/2024 a 30/04/2025), preservando o conjunto de teste e validação rigorosamente selados para evitar o data leakage.

A partir da inspeção estatística e temporal, formulamos três hipóteses justificadas sobre o comportamento da demanda de bicicletas compartilhadas:

1. Hipótese 1 (Padrão Pendular Diário): A demanda exibe comportamento multimodal com picos acentuados nos horários de deslocamento pendular de trabalho e estudo (por volta das 08h e das 18h) em dias úteis, refletindo a rotina urbana.

2. Hipótese 2 (Sensibilidade Climática e Conforto): Condições meteorológicas severas (como chuva ou chuva_forte) e ventos intensos suprimem drasticamente a utilização do serviço, enquanto dias de tempo limpo favorecem o uso.

3. Hipótese 3 (Não-linearidade Térmica): A temperatura e a umidade operam em conjunto por meio de um limiar ótimo; temperaturas excessivamente baixas ou altas desestimulam o uso ao degradar o conforto térmico do ciclista.

In [ ]:
# Missão 1: Carregamento e Verificação Inicial do Dataset
import pandas as pd

# Carregamento do dataset histórico
df = pd.read_csv('base_alunos.csv', parse_dates=['data_hora'])
df = df.sort_values('data_hora').reset_index(drop=True)

# Divisão temporal obrigatória (Protocolo da AP2)
train_mask = (df['data_hora'] >= '2024-01-01') & (
    df['data_hora'] <= '2025-04-30 23:59:59'
)
val_mask = (df['data_hora'] >= '2025-05-01') & (
    df['data_hora'] <= '2025-08-31 23:59:59'
)

df_train = df[train_mask].copy()
df_val = df[val_mask].copy()

print(f'Dimensões Totais: {df.shape}')
print(f'Treino (01/01/2024 a 30/04/2025): {len(df_train)} registros')
print(f'Validação (01/05/2025 a 31/08/2025): {len(df_val)} registros')

Dimensões Totais: (14616, 6)
Treino (01/01/2024 a 30/04/2025): 11664 registros
Validação (01/05/2025 a 31/08/2025): 2952 registros


Missão 2: Baseline de Regressão com Atributos Originais

Como ponto de partida para o modelamento preditivo, construímos um modelo Baseline utilizando exclusivamente as informações originais fornecidas (temperatura_c, umidade_pct, vento_kmh, condicao_tempo) acrescidas apenas da extração direta da hora do dia (hora), sem qualquer engenharia de atributos avançada ou manipulação de alta complexidade. Utilizamos uma Regressão Ridge regularizada com validação estritamente temporal.

In [ ]:
# Missão 2: Criação do Baseline com Atributos Originais
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import numpy as np

# Extração estritamente básica apenas da hora
df_train_base = df_train.copy()
df_train_base['hora'] = df_train_base['data_hora'].dt.hour
df_val_base = df_val.copy()
df_val_base['hora'] = df_val_base['data_hora'].dt.hour

features_originais = ['temperatura_c', 'umidade_pct', 'vento_kmh', 'hora']
cat_originais = ['condicao_tempo']

preprocessor_base = ColumnTransformer(
    transformers=[
        (
            'num',
            Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
            ]),
            features_originais,
        ),
        (
            'cat',
            Pipeline([
                (
                    'imputer',
                    SimpleImputer(strategy='constant', fill_value='desconhecido'),
                ),
                (
                    'onehot',
                    OneHotEncoder(handle_unknown='ignore', sparse_output=False),
                ),
            ]),
            cat_originais,
        ),
    ]
)

baseline_model = Pipeline([
    ('preprocessor', preprocessor_base),
    ('regressor', Ridge(alpha=1.0, random_state=42)),
])

X_train_base = df_train_base[features_originais + cat_originais]
y_train = df_train['demanda_bicicletas']
X_val_base = df_val_base[features_originais + cat_originais]
y_val = df_val['demanda_bicicletas']

baseline_model.fit(X_train_base, y_train)
preds_base = baseline_model.predict(X_val_base)

print('--- PERFORMANCE DO BASELINE (Atributos Originais) ---')
print(f"MAE: {mean_absolute_error(y_val, preds_base):.2f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_val, preds_base)):.2f}")
print(f"R²: {r2_score(y_val, preds_base):.3f}")

--- PERFORMANCE DO BASELINE (Atributos Originais) ---
MAE: 27.23
RMSE: 33.55
R²: 0.269


Missão 3: Engenharia de Atributos Derivados

Em conformidade com o Capítulo 3 de O Preço da Precisão, a engenharia de atributos é o ato de transmutar dados brutos em variáveis com forte poder explicativo. Desenvolvemos 10 atributos derivados agrupados em dimensões lógicas:

1. Atributos Temporais de Calendário: dia_semana, mes e indicador de fim_de_semana.

2. Transformações Cíclicas Horárias (hora_sen, hora_cos): Mapeiam a ciclicidade de 24 horas em coordenadas trigonométricas para que a proximidade entre as 23h e as 00h seja matematicamente compreendida.

3. Faixas Categóricas do Dia (periodo_dia): Segmenta o dia em blocos operacionais (madrugada, manhã, tarde, noite, noite tardia).

4. Interações Climáticas e Físicas: sensacao_termica (modelo de Steadman combinando temperatura e umidade), indice_conforto (razão térmica) e vento_forte (indicador binário de ventania superior a 20 km/h).

In [ ]:
# Missão 3: Implementação de 10 Atributos Derivados
def criar_atributos(data_df):
  d = data_df.copy()

  # Atributos temporais básicos
  d['hora'] = d['data_hora'].dt.hour
  d['dia_semana'] = d['data_hora'].dt.dayofweek
  d['mes'] = d['data_hora'].dt.month
  d['fim_de_semana'] = d['dia_semana'].isin([5, 6]).astype(int)

  # Atributos cíclicos (seno e cosseno)
  d['hora_sen'] = np.sin(2 * np.pi * d['hora'] / 24)
  d['hora_cos'] = np.cos(2 * np.pi * d['hora'] / 24)

  # Faixas categóricas do período do dia
  d['periodo_dia'] = pd.cut(
      d['hora'],
      bins=[-1, 5, 11, 17, 21, 24],
      labels=['madrugada', 'manha', 'tarde', 'noite', 'noite_tardia'],
      ordered=False,
  )

  # Sensação térmica aproximada
  d['sensacao_termica'] = d['temperatura_c'] - (
      0.55 * (1 - d['umidade_pct'] / 100) * (d['temperatura_c'] - 14.5)
  )

  # Índice de conforto climático
  d['indice_conforto'] = d['temperatura_c'] / (d['umidade_pct'] + 1)

  # Indicador de ventania
  d['vento_forte'] = (d['vento_kmh'] > 20).astype(int)

  return d


df_train_feat = criar_atributos(df_train)
df_val_feat = criar_atributos(df_val)

print(
    f'✅ Atributos derivados criados com sucesso! Total de novas features:'
    f' {df_train_feat.shape[1] - df_train.shape[1]}'
)

✅ Atributos derivados criados com sucesso! Total de novas features: 10


Missão 4: Comparação Controlada (Antes e Depois)

Para avaliar o impacto real da engenharia de atributos sob condições estritamente controladas, aplicamos o mesmo algoritmo (Regressão Ridge) com a mesma estratégia de validação temporal e hiperparâmetros antes e depois da inclusão das 10 novas variáveis. O ganho expressivo de performance comprova que enriquecer o espaço de representação supera a limitação de modelos lineares puros.

In [ ]:
# Missão 4: Comparação do Mesmo Algoritmo (Ridge) Antes e Depois da Engenharia de Atributos
num_features = [
    'temperatura_c',
    'umidade_pct',
    'vento_kmh',
    'hora',
    'dia_semana',
    'mes',
    'fim_de_semana',
    'hora_sen',
    'hora_cos',
    'sensacao_termica',
    'indice_conforto',
    'vento_forte',
]
cat_features = ['condicao_tempo', 'periodo_dia']

preprocessor_completo = ColumnTransformer(
    transformers=[
        (
            'num',
            Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
            ]),
            num_features,
        ),
        (
            'cat',
            Pipeline([
                (
                    'imputer',
                    SimpleImputer(strategy='constant', fill_value='desconhecido'),
                ),
                (
                    'onehot',
                    OneHotEncoder(handle_unknown='ignore', sparse_output=False),
                ),
            ]),
            cat_features,
        ),
    ]
)

ridge_eng_model = Pipeline([
    ('preprocessor', preprocessor_completo),
    ('regressor', Ridge(alpha=1.0, random_state=42)),
])

X_train_feat = df_train_feat[num_features + cat_features]
X_val_feat = df_val_feat[num_features + cat_features]

ridge_eng_model.fit(X_train_feat, y_train)
preds_ridge_eng = ridge_eng_model.predict(X_val_feat)

print(
    '--- COMPARAÇÃO CONTROLADA (Ridge com Atributos Engenheirados) ---'
)
print(f"MAE: {mean_absolute_error(y_val, preds_ridge_eng):.2f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_val, preds_ridge_eng)):.2f}")
print(f"R²: {r2_score(y_val, preds_ridge_eng):.3f}")

--- COMPARAÇÃO CONTROLADA (Ridge com Atributos Engenheirados) ---
MAE: 20.02
RMSE: 24.96
R²: 0.595


Missão 5: Avaliação do Segundo Algoritmo e Comparação de Métricas

Avançando para uma arquitetura não-linear baseada em árvores de decisão, avaliamos o Random Forest Regressor utilizando os atributos engenheirados. Comparando lado a lado com o baseline linear através das métricas fundamentais ($MAE$, $RMSE$ e $R^{2}$), observamos a superioridade do modelo ensemble em capturar interações complexas entre sazonalidade temporal e condições meteorológicas.

In [ ]:
# Missão 5: Avaliação do Segundo Algoritmo (Random Forest)
from sklearn.ensemble import RandomForestRegressor

rf_model = Pipeline([
    ('preprocessor', preprocessor_completo),
    (
        'regressor',
        RandomForestRegressor(
            n_estimators=150, max_depth=15, random_state=42, n_jobs=-1
        ),
    ),
])

rf_model.fit(X_train_feat, y_train)
preds_rf = rf_model.predict(X_val_feat)

print('--- AVALIAÇÃO DO SEGUNDO ALGORITMO (Random Forest) ---')
print(f"MAE: {mean_absolute_error(y_val, preds_rf):.2f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_val, preds_rf)):.2f}")
print(f"R²: {r2_score(y_val, preds_rf):.3f}")

--- AVALIAÇÃO DO SEGUNDO ALGORITMO (Random Forest) ---
MAE: 8.10
RMSE: 10.36
R²: 0.930


Missão 6: Estudo de Ablação (Remoção de Grupos de Atributos)

Para mensurar a contribuição isolada dos atributos criados, executamos um estudo de ablação removendo simultaneamente dois grupos distintos de atributos gerados:

Grupo 1: Componentes trigonométricos cíclicos (hora_sen, hora_cos).

Grupo 2: Interações físico-climáticas (sensacao_termica, indice_conforto).

A reavaliação do modelo sem esses blocos permite quantificar a degradação da performance e validar a importância de cada grupo no ecossistema preditivo.

In [ ]:
# Missão 6: Estudo de Ablação (Remoção de 2 Grupos de Atributos)
num_features_ablacao = [
    f
    for f in num_features
    if f not in [
        'hora_sen',
        'hora_cos',
        'sensacao_termica',
        'indice_conforto',
    ]
]
cat_features_abl = ['condicao_tempo', 'periodo_dia']

preprocessor_abl = ColumnTransformer(
    transformers=[
        (
            'num',
            Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
            ]),
            num_features_ablacao,
        ),
        (
            'cat',
            Pipeline([
                (
                    'imputer',
                    SimpleImputer(strategy='constant', fill_value='desconhecido'),
                ),
                (
                    'onehot',
                    OneHotEncoder(handle_unknown='ignore', sparse_output=False),
                ),
            ]),
            cat_features_abl,
        ),
    ]
)

rf_abl = Pipeline([
    ('preprocessor', preprocessor_abl),
    (
        'regressor',
        RandomForestRegressor(
            n_estimators=150, max_depth=15, random_state=42, n_jobs=-1
        ),
    ),
])

X_train_abl = df_train_feat[num_features_ablacao + cat_features_abl]
X_val_abl = df_val_feat[num_features_ablacao + cat_features_abl]

rf_abl.fit(X_train_abl, y_train)
preds_abl = rf_abl.predict(X_val_abl)

print('--- RESULTADOS DO ESTUDO DE ABLAÇÃO (Remoção de 2 Grupos) ---')
print(f"R² após Ablação: {r2_score(y_val, preds_abl):.3f}")
print(f"RMSE após Ablação: {np.sqrt(mean_squared_error(y_val, preds_abl)):.2f}")
print(f"MAE após Ablação: {mean_absolute_error(y_val, preds_abl):.2f}")

--- RESULTADOS DO ESTUDO DE ABLAÇÃO (Remoção de 2 Grupos) ---
R² após Ablação: 0.929
RMSE após Ablação: 10.48
MAE após Ablação: 8.20


Missão 7: Função de Predição para Registros Inéditos

Finalizando o projeto com foco em usabilidade e produção (conforme o Capítulo 11 de O Preço da Precisão), desenvolvemos uma função encapsulada prever_demanda_inédita(). Esta função recebe registros brutos contendo apenas as cinco colunas originais (data_hora, temperatura_c, umidade_pct, vento_kmh, condicao_tempo), aplica automaticamente todas as transformações de engenharia de atributos sem exigir pré-computações manuais, e entrega as previsões consolidadas no arquivo exigido previsoes_teste.csv.

In [ ]:
# Missão 7: Função de Predição para Registros Inéditos e Geração do CSV Final
# Re-treinamento final do modelo campeão com 100% da base histórica disponível
df_train_full_feat = criar_atributos(df)
X_full = df_train_full_feat[num_features + cat_features]
y_full = df_train_full_feat['demanda_bicicletas']

best_model = rf_model
best_model.fit(X_full, y_full)


def prever_demanda_inedita(df_registros_brutos):
  """Recebe um DataFrame contendo APENAS as 5 entradas originais:

  ['data_hora', 'temperatura_c', 'umidade_pct', 'vento_kmh', 'condicao_tempo']
  e retorna as previsões de demanda sem exigir campos derivados pré-computados.
  """
  # Aplica a engenharia de atributos de forma totalmente automatizada
  df_proc = criar_atributos(df_registros_brutos)
  X_entrada = df_proc[num_features + cat_features]

  # Gera as previsões com o modelo campeão otimizado
  previsoes = best_model.predict(X_entrada)

  # Monta o DataFrame de saída
  resultado = df_registros_brutos[['data_hora']].copy()
  resultado['previsao_demanda'] = np.round(previsoes, 2)
  return resultado


# Executando e gerando o arquivo oficial de teste reservado
df_test = pd.read_csv('teste_features_sem_alvo.csv', parse_dates=['data_hora'])
sub_df = prever_demanda_inedita(df_test)
sub_df.to_csv('previsoes_teste.csv', index=False)

print(" Função testada com sucesso! Arquivo 'previsoes_teste.csv' gerado:")
display(sub_df.head(10))

 Função testada com sucesso! Arquivo 'previsoes_teste.csv' gerado:


,data_hora,previsao_demanda
0,2025-09-01 00:00:00,30.67
1,2025-09-01 01:00:00,51.05
2,2025-09-01 02:00:00,42.74
3,2025-09-01 03:00:00,38.55
4,2025-09-01 04:00:00,61.10
5,2025-09-01 05:00:00,55.96
6,2025-09-01 06:00:00,81.10
7,2025-09-01 07:00:00,127.04
8,2025-09-01 08:00:00,149.33
9,2025-09-01 09:00:00,141.58
